# Technical Intro to Generative AI
## Session 1 -- Solutions Notebook
### LLM Architecture and Model Behavior

**Purpose:** Instructor / self-study reference. Every fill-in, open-ended response, and reflection from the student lab notebook is answered here with a model solution and a brief rationale.

---

### Contents

| Section | Title |
|---------|-------|
| Lab 1   | Tokenization and the Token Pipeline |
| Lab 2   | Context Window and Attention Probing |
| Lab 3   | Sampling Parameter Tuning |
| Lab 4   | Programmatic Failure Detection |
| Wrap-up | Reflection answers and competency guide |

> **Note:** Solutions for code exercises show one correct implementation. Reflection answers are model responses -- student answers will vary based on their domain and project context.


---
## Global Setup

Run this cell before executing any lab section.

In [ ]:
# Install required packages (run once)
# !pip install tiktoken openai numpy

import os
import json
import time
import uuid
import numpy as np
from datetime import datetime
from collections import defaultdict

from openai import OpenAI
client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))

CHAT_MODEL      = 'gpt-4o'
EMBEDDING_MODEL = 'text-embedding-3-small'

print('Setup complete.')
print('Chat model      :', CHAT_MODEL)
print('Embedding model :', EMBEDDING_MODEL)


---
---
# Lab 1 -- Tokenization and the Token Pipeline
**Learning Units:** S1-01 to 04, S1-21


### Lab 1 Setup -- Tokenizer helpers

In [ ]:
import tiktoken

enc = tiktoken.encoding_for_model('gpt-4o')

def tokenize(text, encoder=enc):
    token_ids = encoder.encode(text)
    token_strings = [encoder.decode([t]) for t in token_ids]
    return token_ids, token_strings

def show_tokens(text, encoder=enc):
    ids, strings = tokenize(text, encoder)
    print('Input  :', repr(text))
    print('Count  :', len(ids), 'token(s)')
    print('Tokens :', strings)
    print()

print('Tokenizer ready.')


---
### Exercise 1.1 -- Tokenize and Count  [SOLUTION]

**What this demonstrates:** Small surface changes to text -- punctuation, capitalisation, language -- can meaningfully shift the token count. The model sees token IDs, not characters, so token boundaries determine what the model can reason about precisely.


In [ ]:
# Step 1: Baseline sentence
baseline = 'The transformer attention mechanism scales quadratically with sequence length.'
show_tokens(baseline)


In [ ]:
# Step 2: Variation sweep -- solution with representative results

variations = {
    'With comma after transformer':
        'The transformer, attention mechanism scales quadratically with sequence length.',
    'All words capitalised':
        'The Transformer Attention Mechanism Scales Quadratically With Sequence Length.',
    'Domain-specific term substituted':
        'The multi-head self-attention mechanism scales quadratically with sequence length.',
    'Translated to French':
        'Le mecanisme d attention du transformeur evolue de maniere quadratique avec la longueur de la sequence.',
}

baseline_ids, _ = tokenize(baseline)
print(f"{'Variation':<45} {'Tokens':>6}  {'Delta':>6}")
print('-' * 60)
print(f"{'[Baseline]':<45} {len(baseline_ids):>6}  {'--':>6}")

for label, text in variations.items():
    ids, _ = tokenize(text)
    delta  = len(ids) - len(baseline_ids)
    sign   = '+' if delta > 0 else ''
    print(f"{label:<45} {len(ids):>6}  {sign + str(delta):>6}")


In [ ]:
# Step 3: Model observation

# SOLUTION ANSWER
# The translation to French typically adds the most tokens because French words
# are less frequent in the GPT-4o training distribution than English, so the tokenizer
# splits many French words into 2-3 subword units rather than single tokens.
# Capitalising each word often adds a small number of tokens because capitalised
# forms of words sometimes map to different token IDs than their lowercase counterparts.
# Adding a comma rarely changes the count by more than one because punctuation
# characters typically map to individual tokens.

observation = (
    'The French translation added the most tokens (~1.8x the baseline count). '
    'This is because the GPT-4o tokenizer was trained predominantly on English text, '
    'so French words are represented as multiple subword fragments rather than single tokens. '
    'Capitalisation added 1-2 tokens because some capitalised word forms use a separate token ID. '
    'The comma added exactly 1 token. '
    'Implication for prompting: inputs in non-English languages consume context window budget faster.'
)
print(observation)


---
### Exercise 1.2 -- Domain Term Tokenization  [SOLUTION]

**What this demonstrates:** Technical terms, camelCase identifiers, and acronyms that seem like single words to a human often split into multiple sub-tokens. The model attends to fragments separately, which can reduce reliability when that term is central to the task.


In [ ]:
# Step 1: Tokenize representative domain terms
# These are illustrative examples. Students should substitute terms from their own domain.

domain_terms = [
    'clientEngagementID',
    'retrieval-augmented-generation',
    'EBITDA',
    'transformer',
    'McKinseyGPT',
]

print(f"{'Term':<35} {'Tokens':>6}  {'Breakdown'}")
print('-' * 75)
for term in domain_terms:
    ids, strings = tokenize(term)
    flag = 'MULTI-TOKEN -- review prompting strategy' if len(ids) > 1 else 'single token OK'
    print(f"{term:<35} {len(ids):>6}  {strings}")
    print(f"  {'':>35}  --> {flag}")
    print()


In [ ]:
# Step 2: Implication write-up -- model solution

# SOLUTION ANSWER
# 'clientEngagementID' typically splits into ['client', 'Engagement', 'ID'] or similar.
# Implication: when this string appears in a prompt as a JSON key or instruction target,
# the model processes it as three separate attention units. It will generally still
# associate the full concept, but exact-match tasks -- like 'find the value of
# clientEngagementID in this JSON' -- become less reliable than they would be
# for a single-token key. Strategy: test exact-match extraction reliability for
# multi-token keys and prefer single-token or snake_case variants in schemas
# where possible (e.g., 'engagement_id' instead of 'clientEngagementID').

implications = (
    'Term: clientEngagementID -- splits into multiple subword tokens\n'
    'Implication: the model processes this identifier as separate fragments, '
    'not as an atomic unit. For exact-match extraction tasks I should test whether '
    'the model reliably returns the full camelCase string or occasionally truncates it. '
    'Consider using snake_case keys in schemas to improve single-token alignment.'
)
print(implications)


---
### Exercise 1.3 -- Embedding Similarity  [SOLUTION]

**What this demonstrates:** The model's training distribution encodes meaning, not surface form. Semantically similar sentences score high even when the wording differs. Syntactically parallel sentences with different meanings score lower. This matters for retrieval design.


In [ ]:
def get_embedding(text):
    response = client.embeddings.create(input=text, model=EMBEDDING_MODEL)
    return np.array(response.data[0].embedding)

def cosine_similarity(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print('Embedding helpers ready.')


In [ ]:
# Sentence pairs and expected score ranges
pairs = {
    'Semantically similar': (
        'The model exceeded expectations.',
        'The AI system outperformed benchmarks.',
    ),
    'Syntactically similar, semantically different': (
        'The bank by the river flooded.',
        'The bank approved the loan application.',
    ),
    'Unrelated': (
        'The quarterly report is due on Friday.',
        'The transformer uses positional encoding to preserve word order.',
    ),
}

# Expected approximate ranges based on text-embedding-3-small behaviour:
#   Semantically similar      : 0.85 - 0.95
#   Syntactically sim, diff   : 0.60 - 0.75
#   Unrelated                 : 0.30 - 0.55

print(f"{'Pair Type':<50} {'Similarity':>10}  Expected range")
print('-' * 80)

expected = {
    'Semantically similar': '0.85-0.95',
    'Syntactically similar, semantically different': '0.60-0.75',
    'Unrelated': '0.30-0.55',
}

for label, (s1, s2) in pairs.items():
    emb1 = get_embedding(s1)
    emb2 = get_embedding(s2)
    sim  = cosine_similarity(emb1, emb2)
    print(f"{label:<50} {sim:>10.4f}  {expected[label]}")


In [ ]:
# Step 3: Model solution for the 'surprising result' question

# SOLUTION ANSWER
# The syntactically similar but semantically different pair (both use 'bank')
# typically surprises students because the score is lower than expected.
# A naive mental model might assume the word 'bank' would pull the vectors together,
# but modern embedding models use contextual representations -- the word 'bank'
# in a river-flooding context has a different vector than 'bank' in a loan context.
# This is important for retrieval design: a keyword match on 'bank' would incorrectly
# surface both documents, but an embedding retrieval would correctly separate them.

surprising_result = (
    'Pair: syntactically similar, semantically different (the two bank sentences)\n'
    'Observed score: approximately 0.65 (will vary slightly)\n'
    'Why surprising: both sentences use the word bank, but the embeddings correctly '
    'separate the financial and geographical senses. '
    'Implication: embedding-based retrieval disambiguates polysemous terms; '
    'keyword search does not. For retrieval pipelines this is a meaningful advantage.'
)
print(surprising_result)


---
### Lab 1 -- Reflection Answers  [SOLUTION]

In [ ]:
# Q1: Token boundary implication for a real work case
# SOLUTION ANSWER
# Example answer for a data engineering context:
# 'In a recent pipeline I built, I used the field name OpportunityReferenceNumber
# as a JSON key. The model intermittently truncated it to OpportunityReference in
# the output. After tokenizing the key I found it splits into 5 subword tokens.
# Renaming it to opportunity_id (2 tokens) eliminated the truncation entirely.'

# Q2: Why LLMs cannot count letters
# SOLUTION ANSWER
# 'The model processes text as token IDs, not characters. The letter e inside a
# token is invisible to the attention mechanism -- the model sees the token ID,
# not the characters that make it up. Counting occurrences of e requires inspecting
# sub-token content, which is not a computation the transformer performs. The model
# generates a plausible-looking count by pattern-matching the structure of counting
# answers, not by executing a count.'

# Q3: Schema key tokenization for clientEngagementID
# SOLUTION ANSWER
# 'Yes, you should test it. If the key splits into multiple tokens, the model may
# inconsistently reproduce the full camelCase string, producing output that fails
# JSON schema validation or silently maps to the wrong field. Testing takes two
# minutes and prevents a class of production failures that are hard to diagnose
# without knowing the tokenization behaviour.'

reflections_lab1_solutions = {
    'Q1 - Token boundary implication':
        'If I had tokenized the schema key OpportunityReferenceNumber before building '
        'the extraction pipeline, I would have seen it splits into 5 tokens and '
        'switched to a shorter snake_case key immediately, avoiding intermittent '
        'truncation bugs in production.',

    'Q2 - Why LLMs cannot count letters':
        'The model operates on token IDs. The letter e inside a token is invisible '
        'to the model -- it sees only the token ID, not the characters comprising it. '
        'Character-level operations require sub-token inspection, which the transformer '
        'does not perform. The model generates a plausible count by pattern-matching, '
        'not by executing a count.',

    'Q3 - Schema key tokenization':
        'Yes. A multi-token key is reliably reproduced in simple cases but intermittently '
        'truncated at higher temperatures or in longer outputs. Testing takes two minutes '
        'and eliminates a whole class of hard-to-diagnose schema failures.',
}

for q, a in reflections_lab1_solutions.items():
    print('=' * 60)
    print(q)
    print(a)
    print()


---
---
# Lab 2 -- Context Window and Attention Probing
**Learning Units:** S1-05 to 12


### Lab 2 Setup

In [ ]:
def ask_llm(system_prompt, user_message, temperature=0.0, model=CHAT_MODEL):
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user',   'content': user_message},
        ]
    )
    return response.choices[0].message.content.strip()

print('ask_llm helper ready.')


---
### Exercise 2.1 -- The Lost-in-the-Middle Experiment  [SOLUTION]

**What this demonstrates:** Transformer attention is not uniform across the context. Information at the boundaries of a long input is retrieved more reliably than information positioned in the middle. This is an architectural property, not a prompt quality problem.


In [ ]:
# SOLUTION: build_document_version -- complete implementation

def build_document_version(full_text, target_paragraph, position):
    """
    Return a version of the document with target_paragraph placed at
    'beginning', 'middle', or 'end'.
    Splits the document by double newlines and reinserts the target paragraph
    at the appropriate position.
    """
    paragraphs = [p.strip() for p in full_text.strip().split('\n\n') if p.strip()]
    # Remove the target paragraph from wherever it sits
    remaining = [p for p in paragraphs if target_paragraph.strip() not in p]
    n = len(remaining)

    if position == 'beginning':
        ordered = [target_paragraph] + remaining
    elif position == 'middle':
        mid = n // 2
        ordered = remaining[:mid] + [target_paragraph] + remaining[mid:]
    else:  # end
        ordered = remaining + [target_paragraph]

    return '\n\n'.join(ordered)

print('build_document_version ready.')

# ---- Explanation of key design decisions ----
#
# 1. Split on double newline: most plain-text documents use blank lines as paragraph
#    separators. If your document uses single newlines, change the split delimiter.
#
# 2. Strip and filter: removes empty strings that arise from leading/trailing whitespace.
#
# 3. Remove-then-reinsert: ensures the target paragraph appears exactly once regardless
#    of its original position, giving a clean three-way comparison.
#
# 4. Middle position uses floor division to split non-even paragraph counts fairly.


In [ ]:
# Demonstration with a synthetic document
# Replace DOCUMENT_TEXT and TARGET_PARAGRAPH with your own material in the student version.

SAMPLE_DOC = "\n\n".join([
    f"Section {i}: This is paragraph {i} of the sample document. "
    "It contains background information relevant to the overall topic."
    for i in range(1, 16)  # 15 filler paragraphs
])

TARGET_PARAGRAPH = (
    "The project budget was approved at 4.7 million USD on March 12 2023, "
    "with Sarah Chen named as the accountable executive."
)

TARGET_FACT_QUESTION = "What was the approved project budget and who was named accountable executive?"

DOCUMENT_TEXT = SAMPLE_DOC

versions = {
    'beginning': build_document_version(DOCUMENT_TEXT, TARGET_PARAGRAPH, 'beginning'),
    'middle':    build_document_version(DOCUMENT_TEXT, TARGET_PARAGRAPH, 'middle'),
    'end':       build_document_version(DOCUMENT_TEXT, TARGET_PARAGRAPH, 'end'),
}

for name, doc in versions.items():
    word_count = len(doc.split())
    target_pos = doc.find(TARGET_PARAGRAPH[:30])
    print(f"{name:<12}: {word_count} words  target starts at char {target_pos}")


In [ ]:
# Run one trial per version and show results
# (The interactive multi-trial loop from the student notebook is collapsed here
# to a single non-interactive pass for the solutions reference.)

SYSTEM = 'Answer only from the provided document. Be specific and concise.'

print('Running one trial per position version...\n')

for position, doc_text in versions.items():
    user_msg = f"{doc_text}\n\n---\nQuestion: {TARGET_FACT_QUESTION}"
    answer   = ask_llm(SYSTEM, user_msg)
    print(f"[{position.upper()}]")
    print(f"  Answer: {answer[:200]}")
    print()

# EXPECTED PATTERN (may vary by model version):
# - Beginning position: correct retrieval on most or all runs
# - Middle position:    occasional incorrect or incomplete retrieval
# - End position:       correct retrieval on most or all runs
# This is the 'lost-in-the-middle' effect documented in transformer long-context research.


---
### Exercise 2.2 -- Attention Dilution vs. Instruction Failure  [SOLUTION]

**What this demonstrates:** Two different root causes produce visually similar failures. The reduced-input test isolates which cause is active, directing the fix to the right layer.


In [ ]:
# Step 1: Reduced input test -- solution

reduced_user_msg = f"{TARGET_PARAGRAPH}\n\n---\nQuestion: {TARGET_FACT_QUESTION}"
reduced_answer   = ask_llm(SYSTEM, reduced_user_msg)

print('Reduced Input Result')
print('-' * 50)
print(reduced_answer)

# EXPECTED: the model answers correctly on the reduced input.
# The fact is present, the context is minimal, and no competing passages exist.


In [ ]:
# Step 2: Diagnosis -- model solution

# SOLUTION ANSWER
# Correct on reduced input, incorrect (or inconsistent) on full input with middle positioning
# --> ATTENTION DILUTION
#
# Reasoning: the passage containing the target fact is present in the full-context version.
# The model has the information. The failure is not a missing passage or a bad instruction --
# it is that the attention mechanism underweights the passage when it is buried among 14 other
# paragraphs in the middle of the input. The fix is structural (reposition the passage),
# not lexical (rewrite the prompt instruction).

diagnosis = 'ATTENTION DILUTION'
reasoning = (
    'The model answered correctly on the reduced input (target paragraph only) '
    'but was inconsistent on the full input with middle positioning. '
    'The passage is present in the full context. The failure is not in the instruction '
    'or the passage itself -- it is that attention weights are distributed across 15 paragraphs '
    'and the middle-positioned passage receives insufficient weight. '
    'Fix: reposition critical information to the beginning or end of the context.'
)

print('Diagnosis:', diagnosis)
print()
print('Reasoning:', reasoning)


In [ ]:
# Step 3: Reposition and confirm improvement

repositioned_doc = build_document_version(DOCUMENT_TEXT, TARGET_PARAGRAPH, 'beginning')
repositioned_msg = f"{repositioned_doc}\n\n---\nQuestion: {TARGET_FACT_QUESTION}"

print('Running 3 trials with target paragraph repositioned to beginning...\n')

correct = 0
for trial in range(1, 4):
    answer = ask_llm(SYSTEM, repositioned_msg)
    # Check if key fact tokens appear in the answer
    is_correct = '4.7' in answer and ('Chen' in answer or 'Sarah' in answer)
    correct += int(is_correct)
    status = 'PASS' if is_correct else 'FAIL'
    print(f"  Trial {trial}: {status} -- {answer[:120]}")

print(f"\nNew retrieval count after repositioning: {correct} / 3")

# EXPECTED: 3/3 correct after repositioning.
# This confirms the diagnosis: the information was always present; the issue was positional.


---
### Lab 2 -- Reflection Answers  [SOLUTION]

In [ ]:
# SOLUTION ANSWERS

reflections_lab2_solutions = {

    'Q1 - Context positioning strategy':
        'I would place the most critical clauses -- indemnification, liability caps, '
        'termination rights -- at the beginning of the context passed to the model, '
        'and repeat any clause whose answer must be in the output at the end as well. '
        'Clauses that provide background or definitions but are not directly queried '
        'can remain in the middle, since the model does not need to retrieve them with '
        'high precision.',

    'Q2 - Reframing not reading carefully':
        'That description is inaccurate and will lead to the wrong fix. '
        'The model is not reading sequentially -- it is computing attention weights '
        'across all token positions simultaneously. The passage in the middle is not '
        'being skipped; it is being underweighted relative to passages at the boundaries. '
        'A better description is: the model is underweighting middle-context information '
        'due to attention dilution, which is an architectural property, not a carefulness failure. '
        'The fix is to restructure the context, not to rewrite the instruction.',

    'Q3 - Use case context budget':
        'A document comparison tool that sends two 15-page financial reports side by side '
        'would have a context budget of approximately 15000 tokens per document '
        '(roughly 750 words per page times 15 pages at ~1.3 tokens per word), '
        'plus a system prompt of ~500 tokens and output budget of ~1000 tokens, '
        'totaling approximately 31000 tokens -- well within a 200K window but '
        'requiring careful attention to which document sections are positioned where.',
}

for q, a in reflections_lab2_solutions.items():
    print('=' * 60)
    print(q)
    print(a)
    print()


---
---
# Lab 3 -- Sampling Parameter Tuning
**Learning Units:** S1-13 to 16, S1-22


### Lab 3 Setup

In [ ]:
EXTRACTION_SYSTEM = (
    'Extract the requested fields. Return ONLY valid JSON. '
    'No preamble, explanation, or markdown fences.'
)

EXTRACTION_USER = '''
From the following text, extract these fields:
- project_name   (string)
- budget_usd     (integer)
- due_date       (string, format YYYY-MM-DD)
- owner          (string)

Text:
The Orion Analytics initiative, led by Sarah Chen, has been allocated a budget of
$2,400,000 with a delivery deadline of September 30, 2025.
'''

REQUIRED_KEYS  = ['project_name', 'budget_usd', 'due_date', 'owner']
EXPECTED_TYPES = {
    'project_name': str,
    'budget_usd':   int,
    'due_date':     str,
    'owner':        str,
}

def run_extraction(temperature, top_p=1.0):
    raw = ask_llm(EXTRACTION_SYSTEM, EXTRACTION_USER, temperature=temperature)
    try:
        parsed = json.loads(raw)
        missing = [k for k in REQUIRED_KEYS if k not in parsed]
        if missing:
            return raw, False, f'Missing keys: {missing}'
        type_errors = [
            f"{k} should be {EXPECTED_TYPES[k].__name__}, got {type(parsed[k]).__name__}"
            for k in REQUIRED_KEYS if not isinstance(parsed[k], EXPECTED_TYPES[k])
        ]
        if type_errors:
            return raw, False, f'Type errors: {type_errors}'
        return raw, True, None
    except json.JSONDecodeError as e:
        return raw, False, f'JSON parse error: {e}'

print('Lab 3 setup complete.')


---
### Exercise 3.1 -- Temperature Sweep  [SOLUTION]

**What this demonstrates:** Temperature directly controls format compliance for structured output tasks. The compliance threshold is the temperature above which at least one trial fails. For JSON extraction, this is typically between 0.3 and 0.6.


In [ ]:
temperatures   = [0.0, 0.2, 0.5, 0.8, 1.2]
sweep_results  = {}

for temp in temperatures:
    trials = []
    for i in range(3):
        raw, passed, error = run_extraction(temperature=temp)
        trials.append({'passed': passed, 'error': error, 'raw': raw})
    sweep_results[temp] = trials
    pass_count  = sum(t['passed'] for t in trials)
    status      = 'OK ' * pass_count + 'FAIL' * (3 - pass_count)
    first_error = next((t['error'] for t in trials if not t['passed']), 'none')
    print(f"  temp={temp:<5} {pass_count}/3 pass   first error: {first_error}")


In [ ]:
# Identify the compliance threshold
threshold_temp = None

for temp in temperatures:
    pass_count = sum(t['passed'] for t in sweep_results[temp])
    if pass_count < 3 and threshold_temp is None:
        threshold_temp = temp

print(f'Compliance threshold temperature: {threshold_temp}')
print()

# SOLUTION EXPLANATION
# For this extraction task the threshold is typically 0.5 or 0.8.
# At temperature 0.0 the output is fully deterministic and consistently valid JSON.
# As temperature rises, the model begins sampling lower-probability tokens.
# The first failures for a structured task usually manifest as:
#   (a) the budget_usd field returning a string like '2400000' instead of integer 2400000
#   (b) a markdown fence appearing around the JSON block
#   (c) a preamble sentence appearing before the opening brace
# These are all sampling effects, not prompt failures. The fix is lower temperature.
print('Typical threshold: 0.5 to 0.8 for JSON extraction tasks.')
print('Failures typically manifest as: type mismatch on numeric fields, '
      'markdown fences, or preamble text before the JSON block.')


In [ ]:
# Inspect the first failing output
for temp, trials in sweep_results.items():
    for i, trial in enumerate(trials):
        if not trial['passed']:
            print(f'First failure at temp={temp}, trial {i+1}')
            print(f'Error type : {trial["error"]}')
            print(f'Raw output :\n{trial["raw"]}')
            break
    else:
        continue
    break
else:
    print('No failures detected at any tested temperature.')
    print('This is possible at lower model versions or with tighter default settings.')
    print('Try temperature 1.5 or a more complex extraction schema to surface failures.')


---
### Exercise 3.2 -- Top-p Comparison  [SOLUTION]

**What this demonstrates:** For short structured outputs, top-p has a smaller effect than temperature. Compliance differences across top-p values are usually visible only in longer outputs or at temperatures near the compliance threshold.


In [ ]:
# Use the highest fully-compliant temperature from Exercise 3.1
# Update compliant_temp based on your sweep results
compliant_temp = 0.2  # conservative choice; adjust based on your sweep

top_p_values  = [0.85, 0.90, 0.95, 1.0]
top_p_results = {}

print(f'Testing top-p values at temperature={compliant_temp}\n')

for tp in top_p_values:
    trials = []
    for i in range(3):
        response = client.chat.completions.create(
            model=CHAT_MODEL,
            temperature=compliant_temp,
            top_p=tp,
            messages=[
                {'role': 'system', 'content': EXTRACTION_SYSTEM},
                {'role': 'user',   'content': EXTRACTION_USER},
            ]
        )
        raw    = response.choices[0].message.content.strip()
        passed = False
        try:
            parsed = json.loads(raw)
            passed = all(k in parsed for k in REQUIRED_KEYS)
        except json.JSONDecodeError:
            pass
        trials.append({'passed': passed, 'raw': raw})
    pass_count         = sum(t['passed'] for t in trials)
    top_p_results[tp]  = trials
    print(f'  top_p={tp}  {pass_count}/3 pass')

# SOLUTION EXPLANATION
# For a short extraction task like this, top-p differences are usually minimal at
# compliant temperatures. All four values typically produce 3/3 pass.
# The practical rule: for structured output, temperature is the primary control;
# top-p is a secondary fine-tuning lever most useful for longer prose generation.


In [ ]:
# Final parameter configuration card -- model solution

# SOLUTION: reasoning for each choice
# temperature = 0.0  : eliminates all sampling variance for structured output;
#                      the model always selects the highest-probability token,
#                      producing deterministic and schema-compliant JSON.
# top_p = 1.0        : no nucleus restriction needed when temperature is 0.0;
#                      the distribution is already concentrated on the top token.
# frequency_penalty  : leave at 0.0; repetition is not a concern for short JSON.

final_config = {
    'task_type':               'JSON field extraction (4 fields, short input)',
    'temperature':             0.0,
    'top_p':                   1.0,
    'frequency_penalty':       0.0,
    'observed_compliance_rate':'3/3 at temperatures 0.0 and 0.2; failures begin at 0.5-0.8',
    'failure_mode_at_higher_temp':
        'Type mismatch on budget_usd (string instead of int); '
        'occasional markdown fences around JSON block; '
        'preamble sentence before opening brace',
    'notes':
        'If the input text changes significantly -- longer context, ambiguous values -- '
        're-run the sweep because the threshold may shift downward.',
}

print('Final Parameter Configuration Card')
print('-' * 50)
for k, v in final_config.items():
    print(f'  {k:<35}: {v}')


---
### Lab 3 -- Reflection Answers  [SOLUTION]

In [ ]:
# SOLUTION ANSWERS

reflections_lab3_solutions = {

    'Q1 - Diagnosing a teammates issue':
        'The first question is: what temperature and top-p are you running? '
        'Before examining the prompt, we need to confirm whether the output variability '
        'is a sampling parameter problem. If the model is running at default temperature '
        '(usually 0.7-1.0) for a structured extraction task, the JSON breakage is almost '
        'certainly a parameter configuration failure, not a prompt failure. '
        'Setting temperature to 0.0 will likely resolve it in under five minutes. '
        'Iterating on the prompt at default temperature is the wrong diagnostic path.',

    'Q2 - Document routing pipeline':
        'I would use temperature 0.0. '
        'A classification label that routes documents to a workflow is a downstream-automated '
        'output: any variability in the label propagates as a silent system failure. '
        'The label space is finite and known in advance, so there is no benefit to sampling '
        'diversity -- the correct label is deterministic given the input. '
        'Running at temperature 0.0 combined with schema validation and retry logic '
        'gives the highest reliable compliance rate for this output category.',
}

for q, a in reflections_lab3_solutions.items():
    print('=' * 60)
    print(q)
    print(a)
    print()


---
---
# Lab 4 -- Programmatic Failure Detection
**Learning Units:** S1-17 to 20


### Lab 4 Setup

In [ ]:
LOG_STORE = []
print('Log store initialised.')


---
### Exercise 4.1 -- Build a Structured Logger  [SOLUTION]

**What this demonstrates:** Structured logging is the prerequisite for every downstream quality check. Without capturing prompt, parameters, token counts, and latency on every call, you cannot diagnose patterns across runs.


In [ ]:
# SOLUTION: logged_llm_call with all TODO items completed

def logged_llm_call(system_prompt, user_message,
                    temperature=0.0, top_p=1.0,
                    model=CHAT_MODEL, log_store=LOG_STORE):
    """
    Send a chat completion request and append a structured log entry.
    Returns (raw_output, log_entry).

    Logged fields:
    - id            : unique UUID for this call
    - timestamp     : UTC ISO format
    - model         : model identifier string
    - temperature   : sampling temperature used
    - top_p         : nucleus sampling threshold used
    - input_tokens  : token count of prompt (from usage object)  <-- completed
    - output_tokens : token count of completion                  <-- completed
    - latency_ms    : wall-clock time for the API call in ms     <-- completed
    - system_prompt : full system prompt text
    - user_message  : full user message text
    - raw_output    : full raw response string
    """
    start_time = time.time()

    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        top_p=top_p,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user',   'content': user_message},
        ]
    )

    # Solution for the three TODO items:
    latency_ms    = round((time.time() - start_time) * 1000, 1)   # wall-clock ms
    raw_output    = response.choices[0].message.content.strip()
    input_tokens  = response.usage.prompt_tokens                   # from API usage object
    output_tokens = response.usage.completion_tokens               # from API usage object

    log_entry = {
        'id':            str(uuid.uuid4()),
        'timestamp':     datetime.utcnow().isoformat(),
        'model':         model,
        'temperature':   temperature,
        'top_p':         top_p,
        'input_tokens':  input_tokens,
        'output_tokens': output_tokens,
        'latency_ms':    latency_ms,
        'system_prompt': system_prompt,
        'user_message':  user_message,
        'raw_output':    raw_output,
    }

    log_store.append(log_entry)
    return raw_output, log_entry

print('logged_llm_call ready.')


In [ ]:
# Run 3 calls and display log summary
for i in range(3):
    raw, entry = logged_llm_call(
        system_prompt=EXTRACTION_SYSTEM,
        user_message=EXTRACTION_USER,
        temperature=0.0,
    )
    print(f"Call {i+1}: id={entry['id'][:8]}  "
          f"in={entry['input_tokens']} tok  "
          f"out={entry['output_tokens']} tok  "
          f"latency={entry['latency_ms']} ms")

print(f'\nTotal entries in LOG_STORE: {len(LOG_STORE)}')


In [ ]:
# Preview a log entry (truncated for readability)
sample = LOG_STORE[-1].copy()
sample['system_prompt'] = sample['system_prompt'][:80] + '...'
sample['user_message']  = sample['user_message'][:80]  + '...'
sample['raw_output']    = sample['raw_output'][:80]    + '...'
print(json.dumps(sample, indent=2))


---
### Exercise 4.2 -- Automated Quality Checks  [SOLUTION]

**What this demonstrates:** Three layers of automated checking cover the main failure modes for a structured extraction pipeline: schema validity, output length bounds, and format compliance (no preamble text).


In [ ]:
# Check 1: JSON schema validation (provided in student notebook -- shown here for reference)
def check_json_schema(raw_output, required_keys=REQUIRED_KEYS,
                      expected_types=EXPECTED_TYPES):
    try:
        parsed = json.loads(raw_output)
    except json.JSONDecodeError as e:
        return False, f'JSON parse error: {e}'
    missing = [k for k in required_keys if k not in parsed]
    if missing:
        return False, f'Missing keys: {missing}'
    type_errors = [
        f"'{k}' expected {expected_types[k].__name__}, got {type(parsed[k]).__name__}"
        for k in required_keys
        if k in parsed and not isinstance(parsed[k], expected_types[k])
    ]
    if type_errors:
        return False, f'Type mismatch: {type_errors}'
    return True, None


# Check 2: Output token ceiling
MAX_OUTPUT_TOKENS = 100

def check_output_token_ceiling(log_entry, max_tokens=MAX_OUTPUT_TOKENS):
    actual = log_entry['output_tokens']
    if actual > max_tokens:
        return False, f'output_tokens={actual} exceeds ceiling of {max_tokens}'
    return True, None


# Check 3 -- SOLUTION: no preamble detection
# RATIONALE: The model is instructed to return only JSON with no preamble.
# If the output does not begin with '{', the model added explanation text before
# the JSON block. This causes json.loads to fail even if the JSON itself is valid.
# Detecting it separately from the parse error gives a more actionable error message.

def check_no_preamble(raw_output):
    stripped = raw_output.strip()
    if not stripped.startswith('{'):
        preview = stripped[:60].replace('\n', ' ')
        return False, f'Output does not start with curly brace. Preview: {preview}'
    return True, None


# Alternative Check 3 options (instructors can show any of these as examples):
#
# def check_no_model_disclaimer(raw_output):
#     phrases = ['As an AI', 'I cannot', 'I am unable', 'Note:']
#     for phrase in phrases:
#         if phrase.lower() in raw_output.lower():
#             return False, f'Prohibited phrase detected: {phrase}'
#     return True, None
#
# def check_date_format(raw_output):
#     import re
#     try:
#         parsed = json.loads(raw_output)
#         due_date = parsed.get('due_date', '')
#         if not re.match(r'^[0-9]{4}-[0-9]{2}-[0-9]{2}$', str(due_date)):
#             return False, f'due_date format incorrect: {due_date}'
#         return True, None
#     except Exception as e:
#         return False, f'Could not parse output: {e}'

print('All three check functions defined.')


In [ ]:
# Run all checks over the current log store
print(f"{'ID':<10} {'Schema':^8} {'Tokens':^8} {'No Preamble':^12} {'Overall'}")
print('-' * 52)

def fmt(passed): return 'PASS' if passed else 'FAIL'

for entry in LOG_STORE:
    s_pass, s_err = check_json_schema(entry['raw_output'])
    t_pass, t_err = check_output_token_ceiling(entry)
    p_pass, p_err = check_no_preamble(entry['raw_output'])
    overall       = 'PASS' if all([s_pass, t_pass, p_pass]) else 'FAIL'
    print(f"{entry['id'][:8]:<10} {fmt(s_pass):^8} {fmt(t_pass):^8} {fmt(p_pass):^12} {overall}")
    for err in [s_err, t_err, p_err]:
        if err:
            print(f'  Error: {err}')


In [ ]:
# Build a larger sample across temperatures for pattern analysis
print('Building multi-temperature sample for Exercise 4.3...\n')

for temp in [0.0, 0.2, 0.5, 0.8, 1.2]:
    for _ in range(3):
        logged_llm_call(EXTRACTION_SYSTEM, EXTRACTION_USER, temperature=temp)

print(f'Total log entries: {len(LOG_STORE)}')


---
### Exercise 4.3 -- Failure Pattern Analysis  [SOLUTION]

**What this demonstrates:** Grouping failures by temperature reveals whether the failure distribution is concentrated (sampling problem) or random (prompt or schema problem). Each pattern implies a different fix.


In [ ]:
# Group failures by temperature
failures_by_temp = defaultdict(int)
totals_by_temp   = defaultdict(int)

for entry in LOG_STORE:
    temp   = entry['temperature']
    passed, _ = check_json_schema(entry['raw_output'])
    totals_by_temp[temp]   += 1
    if not passed:
        failures_by_temp[temp] += 1

print(f"{'Temperature':>12}  {'Failures':>8}  {'Total':>6}  {'Fail Rate':>9}  Distribution")
print('-' * 65)

for temp in sorted(totals_by_temp):
    total    = totals_by_temp[temp]
    failures = failures_by_temp[temp]
    rate     = failures / total if total else 0
    bar_f    = 'X' * failures
    bar_p    = 'o' * (total - failures)
    print(f"{temp:>12.1f}  {failures:>8}  {total:>6}  {rate:>8.0%}   {bar_f}{bar_p}")


In [ ]:
# Interpret the pattern -- model solution

# SOLUTION ANSWER
# The expected pattern for a JSON extraction task:
#   temp 0.0: 0 failures
#   temp 0.2: 0 failures
#   temp 0.5: 0-1 failures
#   temp 0.8: 1-2 failures
#   temp 1.2: 2-3 failures
#
# This is a CONCENTRATED pattern (failures increase with temperature).
# The implication is that the failures are a sampling parameter problem.
# The model is correctly instructed and the passage is correct -- at low temperature
# it reliably produces valid JSON. Higher temperatures sample lower-probability tokens
# which include format deviations the model learned from training examples.
# Fix: set temperature to 0.0 for this output type. No prompt changes needed.

pattern_interpretation = (
    'PATTERN TYPE: CONCENTRATED AT HIGH TEMPERATURE\n\n'
    'Evidence: Zero failures at temperature 0.0 and 0.2; failures appear at 0.5+ '
    'and increase monotonically with temperature.\n\n'
    'Implication: this is a sampling parameter problem, not a prompt or schema problem. '
    'The model knows the correct output format -- at low temperature it reproduces it '
    'reliably. Higher temperatures introduce token-level variation that breaks JSON validity. '
    'Fix: set temperature to 0.0 for this output type. '
    'A randomly distributed failure pattern would instead suggest a prompt or schema issue.'
)

print(pattern_interpretation)


---
### Exercise 4.4 -- Human Review Scoping  [SOLUTION]

**What this demonstrates:** Human review is not applied uniformly. It is targeted to output categories where the failure consequence is highest and automated checks are insufficient.


In [ ]:
# SOLUTION: completed human review scoping table

review_scope_solution = [
    {
        'output_category':
            'LLM classifies document and routes to automated workflow',
        'primary_failure':
            'Schema or format failure propagates silently through the pipeline',
        'review_strategy':
            'Schema validation on every output; retry logic with temperature=0.0 '
            'on malformed responses; human review of sampled outputs (5-10 percent) '
            'and of any output flagged by automated checks. '
            'Do not review every output -- automated checks handle the bulk.',
    },
    {
        'output_category':
            'LLM summarises findings delivered to client without independent verification',
        'primary_failure':
            'Hallucination: model generates plausible but fabricated claims or citations',
        'review_strategy':
            'Retrieval-grounded generation: every factual claim must map to a source passage. '
            'Human reviewer with domain knowledge verifies claim-to-source mapping '
            'before the output leaves the system. '
            'This review cannot be automated because it requires domain judgment. '
            'Resource and schedule it before the system goes live.',
    },
    {
        'output_category':
            'LLM generates chain-of-thought reasoning steps used internally by a developer',
        'primary_failure':
            'Reasoning gap: arithmetic or multi-step logic error hidden in plausible prose',
        'review_strategy':
            'Developer spot-check of reasoning steps on a sample basis. '
            'Use tool calling (code interpreter) for any arithmetic within the chain of thought '
            'rather than relying on the model to compute. '
            'Full human review of every reasoning step is not required here -- '
            'the output is internal and a developer is already in the loop.',
    },
]

for i, row in enumerate(review_scope_solution, 1):
    print(f'Output Category {i}: {row["output_category"]}')
    print(f'  Primary failure  : {row["primary_failure"]}')
    print(f'  Review strategy  : {row["review_strategy"]}')
    print()


---
### Lab 4 -- Reflection Answers  [SOLUTION]

In [ ]:
# SOLUTION ANSWERS

reflections_lab4_solutions = {

    'Q1 - Plausible fake citation':
        'The correct next step is to verify the citation against the source material '
        'before the output is delivered. '
        'Adjusting the prompt is not the primary fix because hallucination is architectural: '
        'the model generates citation-shaped token sequences when a citation is expected '
        'at that position in the output, regardless of how precisely the prompt instructs it. '
        'The structural mitigations are: (1) retrieval-grounded generation where the model '
        'is only permitted to cite passages provided in its context, and (2) a human review '
        'step that maps each claim to its source before delivery. '
        'Prompt refinement may reduce hallucination frequency but cannot eliminate it '
        'for a factual output that must be verifiably correct.',

    'Q2 - Patterned vs random failures':
        'This is a patterned failure. Ten of twelve failures concentrate on inputs '
        'longer than 800 tokens. That is an 83 percent concentration rate in one '
        'input length band, which is not consistent with random distribution. '
        'The pattern indicates a context or attention issue: the model is more likely '
        'to fail when the input is long, which is consistent with attention dilution '
        'or context window management problems. '
        'The investigation should focus on input structure and chunking strategy for '
        'long inputs, not on prompt wording or parameter configuration.',

    'Q3 - Three mitigations in one sentence each':
        '(a) Hallucination: use retrieval-grounded generation so every factual claim '
        'is anchored to a source passage provided in the context. '
        '(b) Arithmetic errors: route all computation to a code interpreter tool '
        'rather than asking the model to calculate. '
        '(c) Overconfident outputs: include an explicit uncertainty elicitation '
        'instruction in the system prompt, such as: if you are not certain, say so.',
}

for q, a in reflections_lab4_solutions.items():
    print('=' * 60)
    print(q)
    print(a)
    print()


---
---
# Session 1 -- Competency Checklist Guide

The table below describes what a confident answer looks like for each competency item, and what follow-up work is appropriate for partial or no-confidence responses.

| # | Competency | Confident looks like | If partial or needs work |
|---|-----------|---------------------|-------------------------|
| 1 | Stochastic vs deterministic | Can name: sampling, temperature, non-referential-transparency | Re-read LU-S1-01; run temp=0 vs temp=1 side by side |
| 2 | Tokenize and interpret | Can tokenize any string and explain boundary implications | Repeat Exercise 1.2 with 5 new domain terms |
| 3 | Trace token to output | Can describe each stage without notes | Re-read LU-S1-03; trace one real API call step by step |
| 4 | Predict 3 failure types | Can name: context limit, tokenization, throughput | Apply LU-S1-04 to one current integration |
| 5 | Lost-in-the-middle | Can describe the experiment and expected result | Repeat Exercise 2.1 with a different document |
| 6 | Attention dilution vs instruction | Can run the reduced-input diagnostic | Repeat Exercise 2.2 on a failing integration |
| 7 | Context positioning | Has a concrete strategy for own use case | Write out the strategy for one real project |
| 8 | Transformer strengths and weaknesses | Can give two examples of each | Review LU-S1-12 strength/weakness table |
| 9 | Temperature effect prediction | Can predict before running | Add prediction column to Exercise 3.1 table |
| 10 | Parameter sweep | Can run the sweep and read the threshold | Repeat Exercise 3.1 with a different prompt type |
| 11 | Document config | Has a filled-in config card for one real task | Complete the card for a task you currently own |
| 12 | Sequence tuning by risk | Can rank: downstream-auto, client-facing, internal | Apply to the three output types in Exercise 4.4 |
| 13 | Failure mode mechanisms | Can explain all three without looking | Re-read LU-S1-18; write one sentence per mechanism |
| 14 | Structured logger | Built and tested the logger in Exercise 4.1 | Re-implement from scratch without the solution |
| 15 | Automated checks | Wrote a third check of own design | Add two more checks for a different output type |
| 16 | Human review scoping | Completed Exercise 4.4 with defensible reasoning | Apply the scoping framework to one live deployment |


---
### Take-Home Challenge -- Model Answer Structure  [SOLUTION]

In [ ]:
# This is a model brief for a hypothetical document analysis integration.
# Students should substitute their own integration details.

take_home_brief_solution = """
Technical Brief -- Contract Clause Extraction Pipeline

Integration:
An LLM-powered tool that extracts liability caps, indemnification clauses,
and termination rights from vendor contracts and outputs them as structured JSON
for import into a risk tracking database.

1. Context Window Strategy:
Input contracts average 25 pages (~12000 tokens). The system prompt is ~600 tokens
and the output schema is ~300 tokens, leaving approximately 11000 tokens for the
contract body within a 128K context window.
Strategy: the three target clause types are extracted using a chunking approach --
the table of contents is sent first to locate the relevant sections, then only those
sections are sent for extraction. Critical clauses are always positioned in the first
third of the per-section context to avoid middle-position attention degradation.

2. Parameter Configuration Rationale:
temperature = 0.0, because the output is a structured JSON object that feeds an
automated database import. Any sampling variance propagates as a silent schema
failure. The extraction task is deterministic given the input: there is no benefit
to diversity sampling.
top_p = 1.0, because at temperature 0.0 the distribution is already concentrated
and nucleus restriction provides no additional benefit.
frequency_penalty = 0.0, because JSON output is short and repetition is not a risk.
Tested compliance rate: 3/3 across 20 representative contract samples.

3. Failure Detection Approach:
Every API call is logged with input/output token counts, latency, and raw output.
Three automated checks run on every response: JSON schema validation against the
output schema, output token ceiling of 200 (excessively long outputs signal the
model added explanation text), and a check that the response contains no preamble
before the opening brace.
Human review: a lawyer with contract expertise reviews a 10 percent random sample
of outputs weekly, plus all outputs flagged by automated checks. The review verifies
that extracted clause text matches the source document verbatim.
Review burden will be reassessed after the first four weeks of production data.
"""

print(take_home_brief_solution)


---
*Session 1 Solutions Notebook -- Technical Intro to Generative AI -- McKinsey Internal Training*

*For instructor and self-study use. Do not distribute to students before lab completion.*
